In [1]:
import os

import numpy as np
import rasterio
import yaml

from rasterio.enums import Resampling
from rasterio.warp import reproject
from pathlib import Path

In [2]:
os.chdir("..")

In [3]:
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [4]:
# project paths
data_dir = Path(config["data_dir"])
data_dir_raw = data_dir / "raw"
aoi_path = data_dir / "aoi.geojson"

band_paths = [path for path in data_dir_raw.rglob("*.jp2")]
bands_dir = list(set([band.parent for band in band_paths]))[0]

In [5]:
scale_factor = 10000

### Helper functions

In [6]:
# load bands
def load_band(path):
    with rasterio.open(path, "r") as src:
        return {
            "data": src.read(1),
            "metadata": {
                "band_path": path,
                "profile": src.profile,
                "crs": src.crs,
                "transform": src.transform,
                "res": src.res,
                "nodata": src.nodata
            }
        }

In [7]:
# scale digital numbers to surface reflectance
def scale_reflectance(band, scale_factor):
    return band / scale_factor

In [8]:
# resampling
ref_band = load_band(band_paths[1])

def resample_to_10m(src):
    src_band = bands[src]
    dest_10m = np.empty(ref_band["data"].shape, dtype=ref_band["data"].dtype)
    
    reproject(
        source=src_band["data"],
        destination=dest_10m,
        src_transform=src_band["metadata"]["transform"],
        src_crs=src_band["metadata"]["crs"],
        #
        dst_transform=ref_band["metadata"]["transform"],
        dst_crs=ref_band["metadata"]["crs"],
        resampling = Resampling.nearest if src == "SCL" else Resampling.bilinear
    )
    return dest_10m

### Data preparation

In [9]:
bands = {}

for band in band_paths:
    b = band.name[-11:-8]
    bands[b] = load_band(band)
    if b not in ["SCL"]:
        bands[b]["data"] = scale_reflectance(bands[b]["data"], scale_factor)
    if band.name[-7:-4] in ["20m", "60m"]:
        bands[b]["data"] = resample_to_10m(b)